# Lab 5: Text Representation

In [1]:
import re

import pandas as pd
import nltk
import gensim
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

True

## Task 1: Cosine Similarity

In [2]:
sentences = ['This is the first document.',
             'This document is the second document.',
             'And this is the third one.',
             'Is this the first document?']

tfidf = TfidfVectorizer()
vectors = tfidf.fit_transform(sentences)

names = [f'S{i}' for i in range(1, len(sentences) + 1)]
for n, s in zip(names, sentences):
    print(f'{n}: {s}')

pd.DataFrame(cosine_similarity(vectors), index=names, columns=names).round(3)

S1: This is the first document.
S2: This document is the second document.
S3: And this is the third one.
S4: Is this the first document?


,S1,S2,S3,S4
S1,1.000,0.647,0.308,1.000
S2,0.647,1.000,0.225,0.647
S3,0.308,0.225,1.000,0.308
S4,1.000,0.647,0.308,1.000


S1 and S4 contain exactly the same words (TF-IDF ignores word order and punctuation), so their similarity is 1.0. S3 is the least similar to the others (lowest pair: S2 and S3 = 0.225).

## Task 2: TF-IDF

In [3]:
docs = ['data science is one of the most important fields of science',
        'this is one of the best data science courses',
        'data scientists analyze data']

tfidf = TfidfVectorizer()
result = tfidf.fit_transform(docs)

df_tfidf = pd.DataFrame(result.toarray(), columns=tfidf.get_feature_names_out(),
                        index=['doc 1', 'doc 2', 'doc 3'])
df_tfidf.round(3)

,analyze,best,courses,data,fields,important,is,most,of,one,science,scientists,the,this
doc 1,0.000,0.0,0.0,0.190,0.321,0.321,0.244,0.321,0.488,0.244,0.488,0.000,0.244,0.0
doc 2,0.000,0.4,0.4,0.236,0.000,0.000,0.304,0.000,0.304,0.304,0.304,0.000,0.304,0.4
doc 3,0.543,0.0,0.0,0.641,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.543,0.000,0.0


In [4]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# the 3 most important words of each document (highest TF-IDF score, stop words ignored)
content_words = [w for w in df_tfidf.columns if w not in ENGLISH_STOP_WORDS]

for name, row in df_tfidf[content_words].iterrows():
    top = row.sort_values(ascending=False).head(3)
    print(name, '->', ', '.join(f'{w} ({s:.3f})' for w, s in top.items()))

doc 1 -> science (0.488), fields (0.321), important (0.321)
doc 2 -> best (0.400), courses (0.400), science (0.304)
doc 3 -> data (0.641), analyze (0.543), scientists (0.543)


## Task 3: Word2Vec on the Simpsons dataset

In [5]:
df = pd.read_csv('simpsons_script_lines.csv', low_memory=False)
df = df.dropna(subset=['spoken_words'])          # use the 'spoken_words' column
print('Lines:', len(df))

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[0-9]", '', text)
    text = re.sub(r"[)(,”“.’$-]", '', text)
    return text

df['clean_words'] = df['spoken_words'].apply(clean_text)
df[['spoken_words', 'clean_words']].head()

Lines: 131685


,spoken_words,clean_words
0,"No, actually, it was a little of both. Sometim...",no actually it was a little of both sometimes ...
1,Where's Mr. Bergstrom?,where's mr bergstrom?
2,I don't know. Although I'd sure like to talk t...,i don't know although i'd sure like to talk to...
3,That life is worth living.,that life is worth living
4,The polls will be open from now until the end ...,the polls will be open from now until the end ...


In [6]:
tokens = [word_tokenize(line) for line in df['clean_words']]

print('Number of tokenized lines:', len(tokens))
print('Example:', tokens[0])

Number of tokenized lines: 131685
Example: ['no', 'actually', 'it', 'was', 'a', 'little', 'of', 'both', 'sometimes', 'when', 'a', 'disease', 'is', 'in', 'all', 'the', 'magazines', 'and', 'all', 'the', 'news', 'shows', 'it', "'s", 'only', 'natural', 'that', 'you', 'think', 'you', 'have', 'it']


In [7]:
# skip-gram model (seed / workers=1 only make the result reproducible)
Skip_gram_model = gensim.models.Word2Vec(tokens, min_count=1, vector_size=100, window=5, sg=1,
                                         seed=42, workers=1)

print('Vocabulary size:', len(Skip_gram_model.wv))

Vocabulary size: 44128


## Task 4: most_similar()

In [8]:
for word in ['homer', 'marge', 'bart']:
    print(f"Words similar to '{word}':")
    for w, score in Skip_gram_model.wv.most_similar(word):
        print(f'   {w:<15}{score:.4f}')
    print()

Words similar to 'homer':
   abe            0.8843
   marge          0.8530
   bartholomew    0.8331
   bart           0.8280
   eliza          0.8160
   jessica        0.8105
   jay            0.8104
   grampa         0.8090
   laddie         0.8064
   sweetheart     0.8037

Words similar to 'marge':
   abe            0.8656
   homer          0.8530
   sweetie        0.8465
   sweetheart     0.8440
   allison        0.8363
   becky          0.8340
   eliza          0.8324
   honey          0.8314
   midge          0.8249
   jessica        0.8247

Words similar to 'bart':
   lisa           0.8625
   abe            0.8607
   eliza          0.8575
   milhouse       0.8537
   jessica        0.8500
   sweetheart     0.8451
   sweetie        0.8369
   grampa         0.8334
   allison        0.8289
   maggie         0.8281



## Task 5: doesnt_match()

In [9]:
tests = [['jimbo', 'milhouse', 'kearney'],
         ['nelson', 'bart', 'milhouse'],
         ['homer', 'patty', 'selma']]

for words in tests:
    print(f'{words} -> odd one out: {Skip_gram_model.wv.doesnt_match(words)}')

['jimbo', 'milhouse', 'kearney'] -> odd one out: milhouse
['nelson', 'bart', 'milhouse'] -> odd one out: nelson
['homer', 'patty', 'selma'] -> odd one out: homer
